<a target="_blank" href="https://colab.research.google.com/github/bettercodepaul/data-wrangling-praktikum/blob/master/Polars_Part_1.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Polars: The Turbo Boost for Dataframes

In this notebook, we'll get to know Polars. Polars is an extremely fast dataframe library or in-memory query engine. It features extremely parallel execution, cache-efficient algorithms, and an expressive API. This makes it perfect for efficient querying and transformation of data.

Polars is written in Rust, uses Apache Arrow's column-oriented format, and has a Python API.

More information is available here:

- Homepage of Polars: https://www.pola.rs/
- User Guide: https://pola-rs.github.io/polars/user-guide/
- API Reference: https://pola-rs.github.io/polars/py-polars/html/reference/

## Installation + Set-Up

In [2]:
# Download utility module und data
import urllib.request
import os.path
UTILS_URL = "https://github.com/bettercodepaul/data-wrangling-praktikum/raw/master/utils.py"
urllib.request.urlretrieve(UTILS_URL, os.path.basename(UTILS_URL))
from utils import download_data
download_data()

In [3]:
# install libraries
!uv pip install -qr requirements.txt

In [4]:
# import polars
import polars as pl
# output up to 60 characters per column and do not abbreviate floating point numbers
pl.Config(fmt_str_lengths=60, fmt_float="full")

In [5]:
# import exercises and utility functions
from utils import *
from exercises_en import *

## Data Loading

Polars supports different formats when loading data into a dataframe:

- CSV (`read_csv`, `read_csv_batched`)
- Apache Parquet (`read_parquet`)
- Databricks Delta (`read_delta`)
- SQL-Datenbanken (`read_database`, `read_database_uri`)
- JSON (`read_json`, `read_ndjson`)
- Microsoft Excel (`read_excel`)
- Apache OpenOffice (`read_ods`)
- Apache Avro (`read_avro`)
- Apache IPC (`read_ipc`, `read_ipc_stream`)
- Apache Iceberg

We first load a CSV file

In [6]:
# load a CSV file
df = pl.read_csv("spotify-charts-2017-2021-global-top200.csv.gz")
df.head(2) # output the first 2 rows

title,rank,date,artist,url,region,chart,trend,streams
str,i64,str,str,str,str,str,str,i64
"""Starboy""",1,"""2017-01-01""","""The Weeknd, Daft Punk""","""https://open.spotify.com/track/5aAx2yezTd8zXrkmtKl66Z""","""Global""","""top200""","""SAME_POSITION""",3135625
"""Closer""",2,"""2017-01-01""","""The Chainsmokers, Halsey""","""https://open.spotify.com/track/7BKLCZ1jbUBVqRi2FVlTVw""","""Global""","""top200""","""SAME_POSITION""",3015525


The file contains the daily Spotify charts. The following information is included:

- `title`: title of the song
- `rank`: ranking in the charts
- `date`: day on which the chart was compiled
- `artist`: band or artists performing the song
- `url`: URL where the song can be heard on Spotify
- `region`: region or country for which the charts were collected
- `chart`: name or type of the charts
- `trend`: development of the song's ranking compared to the previous day
- `streams`: number of streams of the song on that day

In each column, under the column name, you can see the data type of the column. The date column was read in as a string (`str`), this can be corrected using the `try_parse_dates` option.

In [7]:
# Load data from CSV file and parse date columns
df = pl.read_csv("spotify-charts-2017-2021-global-top200.csv.gz", try_parse_dates=True)
df.head(2) # output the first 2 rows

title,rank,date,artist,url,region,chart,trend,streams
str,i64,date,str,str,str,str,str,i64
"""Starboy""",1,2017-01-01,"""The Weeknd, Daft Punk""","""https://open.spotify.com/track/5aAx2yezTd8zXrkmtKl66Z""","""Global""","""top200""","""SAME_POSITION""",3135625
"""Closer""",2,2017-01-01,"""The Chainsmokers, Halsey""","""https://open.spotify.com/track/7BKLCZ1jbUBVqRi2FVlTVw""","""Global""","""top200""","""SAME_POSITION""",3015525


## Projection (select columns)

If not all columns of a dataframe are needed, certain columns can be selected with the `select` method.

In [8]:
df.select("title", "artist", "url", "streams").head(2)

title,artist,url,streams
str,str,str,i64
"""Starboy""","""The Weeknd, Daft Punk""","""https://open.spotify.com/track/5aAx2yezTd8zXrkmtKl66Z""",3135625
"""Closer""","""The Chainsmokers, Halsey""","""https://open.spotify.com/track/7BKLCZ1jbUBVqRi2FVlTVw""",3015525


## Generalized projection (change or add columns)

With the help of expressions we can change columns or add new columns.

A column can be referenced with the `pl.col` method specifying the column name.

In [9]:
df.select("title", "artist", pl.col("url"), pl.col("streams")).head(2)

title,artist,url,streams
str,str,str,i64
"""Starboy""","""The Weeknd, Daft Punk""","""https://open.spotify.com/track/5aAx2yezTd8zXrkmtKl66Z""",3135625
"""Closer""","""The Chainsmokers, Halsey""","""https://open.spotify.com/track/7BKLCZ1jbUBVqRi2FVlTVw""",3015525


In order to get smaller numbers, we can specify, for example, the number of streams in thousands.

In [10]:
# Division with the "/" operator converts to floating point
df.select(pl.col("title"), pl.col("artist"), pl.col("url"), pl.col("streams")/1000).head(2)

title,artist,url,streams
str,str,str,f64
"""Starboy""","""The Weeknd, Daft Punk""","""https://open.spotify.com/track/5aAx2yezTd8zXrkmtKl66Z""",3135.625
"""Closer""","""The Chainsmokers, Halsey""","""https://open.spotify.com/track/7BKLCZ1jbUBVqRi2FVlTVw""",3015.525


In [11]:
# Alternative with "floordiv", mixed notation with pure column name and pl.col is also possible
df.select("title", "artist", "url", pl.col("streams").floordiv(1000)).head(2)

title,artist,url,streams
str,str,str,i64
"""Starboy""","""The Weeknd, Daft Punk""","""https://open.spotify.com/track/5aAx2yezTd8zXrkmtKl66Z""",3135
"""Closer""","""The Chainsmokers, Halsey""","""https://open.spotify.com/track/7BKLCZ1jbUBVqRi2FVlTVw""",3015


In addition to standard operators such as `+`, `-`, `*` and `/`, a variety of expressions for calculations with numbers is available:

- https://pola-rs.github.io/polars/py-polars/html/reference/expressions/computation.html
- https://pola-rs.github.io/polars/py-polars/html/reference/expressions/operators.html

There are also many functions for the manipulation of strings. These are addressed via their own namespace `str`.

- https://pola-rs.github.io/polars/py-polars/html/reference/expressions/string.html

A selection of commonly used functions for strings:

- `str.starts_with`, `str.ends_with`, `str.contains`
- `str.slice`
- `str.replace`
- `str.to_date`, `str.to_datetime`
- `str.split`
- `str.strip_chars`
- `str.len_chars`

In [12]:
df.select(pl.col("title").str.to_uppercase(), "artist", "url", "streams").head(2)

title,artist,url,streams
str,str,str,i64
"""STARBOY""","""The Weeknd, Daft Punk""","""https://open.spotify.com/track/5aAx2yezTd8zXrkmtKl66Z""",3135625
"""CLOSER""","""The Chainsmokers, Halsey""","""https://open.spotify.com/track/7BKLCZ1jbUBVqRi2FVlTVw""",3015525


In order not to always have to list all columns that are not transformed at all, the function `with_columns` can be used.

In [13]:
# with_columns corresponds to select supplemented by all missing columns
df.with_columns(pl.col("title").str.to_uppercase()).head(2)

title,rank,date,artist,url,region,chart,trend,streams
str,i64,date,str,str,str,str,str,i64
"""STARBOY""",1,2017-01-01,"""The Weeknd, Daft Punk""","""https://open.spotify.com/track/5aAx2yezTd8zXrkmtKl66Z""","""Global""","""top200""","""SAME_POSITION""",3135625
"""CLOSER""",2,2017-01-01,"""The Chainsmokers, Halsey""","""https://open.spotify.com/track/7BKLCZ1jbUBVqRi2FVlTVw""","""Global""","""top200""","""SAME_POSITION""",3015525


So far we have not added any columns. A new column will be created if we specify a name that does not exist yet. We can use the following methods for this:

- `alias` for a completely new name
- `name.prefix`/`name.suffix` to add a prefix/suffix to the existing name

In [14]:
# Extract trackId from the URL
df.select("title", "url").with_columns(pl.col("url").str.slice(len("https://open.spotify.com/track/")).alias("trackId"), pl.col("title").str.to_uppercase().name.suffix("_uppercase")).head(2)

title,url,trackId,title_uppercase
str,str,str,str
"""Starboy""","""https://open.spotify.com/track/5aAx2yezTd8zXrkmtKl66Z""","""5aAx2yezTd8zXrkmtKl66Z""","""STARBOY"""
"""Closer""","""https://open.spotify.com/track/7BKLCZ1jbUBVqRi2FVlTVw""","""7BKLCZ1jbUBVqRi2FVlTVw""","""CLOSER"""


If the entire query becomes too long, it should be bracketed and structured with breaks. In this way, a typical "query pipeline" is created that can be read from top to bottom.

In [15]:
(df
  .select("title", "url")
  .with_columns(
    pl.col("url").str.slice(len("https://open.spotify.com/track/")).alias("trackId"),
    pl.col("title").str.to_uppercase().name.suffix("_uppercase")
  )
  .head(2)
)

title,url,trackId,title_uppercase
str,str,str,str
"""Starboy""","""https://open.spotify.com/track/5aAx2yezTd8zXrkmtKl66Z""","""5aAx2yezTd8zXrkmtKl66Z""","""STARBOY"""
"""Closer""","""https://open.spotify.com/track/7BKLCZ1jbUBVqRi2FVlTVw""","""7BKLCZ1jbUBVqRi2FVlTVw""","""CLOSER"""


We can also use aggregating function like `min`, `max`, `sum`, `mean`, `median`, etc. in `select`, which will give us an aggregation. If we use a column more than once, we have to be careful to give it an appropriate name. Either with `alias` or with `suffix`.

In [16]:
# Determine period for which data is available
df.select(pl.col("date").min().name.suffix("_min"), pl.col("date").max().name.suffix("_max"))

date_min,date_max
date,date
2017-01-01,2021-12-20


## Selection/Filter

With the help of the selection, the data set can be filtered to specific data sets.

For a quick overview the methods `head`, `tail` and `sample` can be used.

In [17]:
# the first two rows
df.head(2)

title,rank,date,artist,url,region,chart,trend,streams
str,i64,date,str,str,str,str,str,i64
"""Starboy""",1,2017-01-01,"""The Weeknd, Daft Punk""","""https://open.spotify.com/track/5aAx2yezTd8zXrkmtKl66Z""","""Global""","""top200""","""SAME_POSITION""",3135625
"""Closer""",2,2017-01-01,"""The Chainsmokers, Halsey""","""https://open.spotify.com/track/7BKLCZ1jbUBVqRi2FVlTVw""","""Global""","""top200""","""SAME_POSITION""",3015525


In [18]:
# the last two rows
df.tail(2)

title,rank,date,artist,url,region,chart,trend,streams
str,i64,date,str,str,str,str,str,i64
"""The Scientist""",199,2018-01-31,"""Coldplay""","""https://open.spotify.com/track/75JFxkI2RXiU7L9VXzMkle""","""Global""","""top200""","""NEW_ENTRY""",533124
"""Stay With Me""",200,2018-01-31,"""Sam Smith""","""https://open.spotify.com/track/5Db9VIdDsN5yu3Eu7CT0i4""","""Global""","""top200""","""NEW_ENTRY""",531654


In [19]:
# two random rows (absolute with parameter "n" or relative with parameter "fraction")
df.sample(n=2)
df.sample(fraction=2/len(df)) # 2/362182 ≈ 0.000006is equivalent here to n=2

title,rank,date,artist,url,region,chart,trend,streams
str,i64,date,str,str,str,str,str,i64
"""I Like Me Better""",166,2018-05-29,"""Lauv""","""https://open.spotify.com/track/1wjzFQodRWrPcQ0AnYnvQ9""","""Global""","""top200""","""MOVE_UP""",630089
"""Me Niego (feat. Ozuna & Wisin)""",34,2018-06-17,"""Reik""","""https://open.spotify.com/track/6jA8JUuPCGYjFcgw0AoM5T""","""Global""","""top200""","""SAME_POSITION""",1315181


The rows with the largest or smallest value in a column can be selected with the functions 'top_k' and 'bottom_k'.

In [20]:
# the most streamed song on Spotify in one day worldwide: Easy On Me by Adele
df.top_k(1, by="streams")

title,rank,date,artist,url,region,chart,trend,streams
str,i64,date,str,str,str,str,str,i64
"""Easy On Me""",1,2021-10-15,"""Adele""","""https://open.spotify.com/track/0gplL1WMoJ6iYaPgMCL0gX""","""Global""","""top200""","""MOVE_UP""",19749704


We can listen to that one, too.

In [21]:
# plays a preview of the song with Spotify. If there are multiple songs in the dataframe, a row number can be specified.
play_song(df.top_k(1, by="streams"))

Rows can be selected precisely using the `filter` method and a Boolean expression. For example, we can select all records of a particular artist.

In [22]:
# two rows for the singer "Adele".
# eq stands for equals
df.filter(pl.col("artist").eq("Adele")).head(2)

title,rank,date,artist,url,region,chart,trend,streams
str,i64,date,str,str,str,str,str,i64
"""Send My Love (To Your New Lover)""",73,2017-01-01,"""Adele""","""https://open.spotify.com/track/4BHzQ9C00ceJxfG16AlNWb""","""Global""","""top200""","""MOVE_UP""",672465
"""Water Under the Bridge""",96,2017-01-01,"""Adele""","""https://open.spotify.com/track/4vb4mFvYsr2h6enhjJsq9Y""","""Global""","""top200""","""MOVE_UP""",565509


An overview of important operators:
- Equal (`==`): `eq`
- Not Equal (`!=`): `ne`.
- Greater Than (`>`, `>=`): `gt`, `ge`
- Less Than (`<`, `<=`): `lt`, `le`
- Between: `is_between`
- Equal to one of a set: `is_in`.

Logical expressions can be linked with:
- conjunction/AND: `&`
- disjunction/OR: `|`
- Contravalence/XOR: `^`
- Negation/NOT: `~`

In [23]:
# two entries for the song "Easy On Me" by Adele with more than 3 million streams in one day
df.filter(pl.col("artist").eq("Adele") & pl.col("title").eq("Easy On Me") & pl.col("streams").gt(3_000_000)).head(2)

title,rank,date,artist,url,region,chart,trend,streams
str,i64,date,str,str,str,str,str,i64
"""Easy On Me""",1,2021-11-01,"""Adele""","""https://open.spotify.com/track/0gplL1WMoJ6iYaPgMCL0gX""","""Global""","""top200""","""SAME_POSITION""",6340943
"""Easy On Me""",1,2021-11-02,"""Adele""","""https://open.spotify.com/track/0gplL1WMoJ6iYaPgMCL0gX""","""Global""","""top200""","""SAME_POSITION""",6458442


Instead of the operators `eq` and `gt` it would also be possible to use the standard Python operators `==` and `>`. But then all logical subexpressions have to be compounded. What you prefer is in the end a matter of taste 😁

In [24]:
df.filter(pl.col("artist").eq("Adele") & pl.col("title").eq("Easy On Me") & pl.col("streams").gt(3_000_000)).head(2)
df.filter((pl.col("artist") == "Adele") & (pl.col("title") == "Easy On Me") & (pl.col("streams") > 3_000_000)).head(2)

title,rank,date,artist,url,region,chart,trend,streams
str,i64,date,str,str,str,str,str,i64
"""Easy On Me""",1,2021-11-01,"""Adele""","""https://open.spotify.com/track/0gplL1WMoJ6iYaPgMCL0gX""","""Global""","""top200""","""SAME_POSITION""",6340943
"""Easy On Me""",1,2021-11-02,"""Adele""","""https://open.spotify.com/track/0gplL1WMoJ6iYaPgMCL0gX""","""Global""","""top200""","""SAME_POSITION""",6458442


For a comparison with a specific date, the date can be generated with the function `pl.date`.

In [25]:
# two entries for May 1, 2017
df.filter(pl.col("date").eq(pl.date(2017, 5, 1))).head(2)

title,rank,date,artist,url,region,chart,trend,streams
str,i64,date,str,str,str,str,str,i64
"""Despacito - Remix""",1,2017-05-01,"""Luis Fonsi, Daddy Yankee, Justin Bieber""","""https://open.spotify.com/track/5CtI0qwDJkDQGwXD1H1cLb""","""Global""","""top200""","""SAME_POSITION""",6360737
"""I'm the One""",2,2017-05-01,"""DJ Khaled, Justin Bieber, Quavo, Chance the Rapper, Lil Wayn…","""https://open.spotify.com/track/72Q0FQQo32KJloivv5xge2""","""Global""","""top200""","""NEW_ENTRY""",5039594


In [26]:
# the ranks 5 to 10 for July 19, 2018
df.filter(pl.col("date").eq(pl.date(2018, 7, 19)) & pl.col("rank").is_between(5, 10))

title,rank,date,artist,url,region,chart,trend,streams
str,i64,date,str,str,str,str,str,i64
"""Lucid Dreams""",5,2018-07-19,"""Juice WRLD""","""https://open.spotify.com/track/0s3nnoMeVWz3989MkNQiRf""","""Global""","""top200""","""SAME_POSITION""",3694745
"""SAD!""",6,2018-07-19,"""XXXTENTACION""","""https://open.spotify.com/track/3ee8Jmje8o58CHK66QrVC2""","""Global""","""top200""","""MOVE_UP""",3657553
"""Taste (feat. Offset)""",7,2018-07-19,"""Tyga""","""https://open.spotify.com/track/5IaHrVsrferBYDm0bDyABy""","""Global""","""top200""","""MOVE_UP""",3551556
"""Solo (feat. Demi Lovato)""",8,2018-07-19,"""Clean Bandit""","""https://open.spotify.com/track/6kPJZM97LwdG9QIsT7khp6""","""Global""","""top200""","""MOVE_UP""",3467576
"""One Kiss (with Dua Lipa)""",9,2018-07-19,"""Calvin Harris""","""https://open.spotify.com/track/7ef4DlsgrMEH11cDZd32M6""","""Global""","""top200""","""MOVE_DOWN""",3446638
"""Don’t Matter To Me""",10,2018-07-19,"""Drake, Michael Jackson""","""https://open.spotify.com/track/36ONiya0OANYknz0GgJmwB""","""Global""","""top200""","""MOVE_DOWN""",3389058


We can also plot daily streams or ranks with a helper function.

In [27]:
some_song_df = df.filter(pl.col("artist").eq("Juice WRLD") & pl.col("title").eq("Lucid Dreams"))

In [28]:
plot_streams(some_song_df)

In [29]:
plot_rank(some_song_df)

## Exercises on projection and selection

You can do the exercise right here in the notebook. For each exercise there is an object (`q1`, `q2`, `q3`, ...) that contains the question, a hint, an answer check and the solution.

In [30]:
# The method "question" prints the question.
q0.question()

Which company created this workshop?


In [32]:
# Then there is always a cell with a hint in which variables the solution should be written.
# Feel free to create more cells to inspect your solution more closely.
awesome_company = "BettercallPaul"

In [33]:
# the method "check" checks a solution
q0.check(awesome_company)

"✅ Wow, first try and you nailed it! You're a natural problem-solver! 🎉👏"

In [34]:
# the "hint" method displays a hint
q0.hint()

It's not BettercallSaul.


In [35]:
# the "solution" method prints the solution
q0.solution()

awesome_company = "BettercallPaul"


Now it's your turn with the real exercises!

### Question 1

In [36]:
q1.question()

Select only the columns "date", "rank", "title" and "artist" from the dataframe "df".


In [37]:
q1_df = df.select("date","rank","title", "artist")

In [38]:
q1.check(q1_df)
#q1.hint()
#q1.solution()

"✅ Wow, first try and you nailed it! You're a natural problem-solver! 🎉👏"

### Question 2

In [39]:
q2.question()

Now select the columns "date", "rank", "title" and "artist", but rename the column "artist" to "performer".


In [41]:
q2_df = df.select("date","rank","title", pl.col("artist").alias("performer"))

In [42]:
q2.check(q2_df)

"✅ Wow, first try and you nailed it! You're a natural problem-solver! 🎉👏"

### Question 3

In [43]:
q3.question()

What is the average of streams in the dataset?


In [44]:
q3_df = df.select(pl.col("streams")).mean()

In [45]:
q3.check(q3_df)

"✅ Wow, first try and you nailed it! You're a natural problem-solver! 🎉👏"

### Question 4

In [47]:
q4.question()

How many times were songs at #1 streamed per day on average, and how many times for the songs at #200?


In [49]:
rank_1 = df.filter(pl.col("rank").eq(1)).select(pl.col("streams")).mean()
rank_200 = df.filter(pl.col("rank").eq(200)).select(pl.col("streams")).mean()

In [50]:
q4.check(rank_1, rank_200)

"✅ Wow, first try and you nailed it! You're a natural problem-solver! 🎉👏"

### Question 5

In [51]:
q5.question()

What song was #1 on New Year's Eve 2017? Also enjoy listening to it.


In [54]:
q5_df = df.filter(pl.col("date").eq(pl.date(2017, 12, 31)) & pl.col("rank").eq(1))

In [55]:
q5.check(q5_df)

'✅ Right on target! You hit the bullseye on your second shot! 🎯👏'

### Question 6

In [56]:
q6.question()


🎄🎅🏻 X-Mas-Showdown 🎅🏻🎄
"Last Christmas" by "Wham!" or "All I Want for Christmas Is You" by "Mariah Carey"???
Filter on the two songs and then plot the streams. Which is your favorite?
    


In [58]:
q6_df = df.filter(pl.col("title").eq("Last Christmas") & pl.col("artist").eq("Wham!") | pl.col("title").eq("All I Want for Christmas Is You") & pl.col("artist").eq("Mariah Carey"))
plot_rank(q6_df)

In [59]:
q6.check(q6_df)

"✅ Wow, first try and you nailed it! You're a natural problem-solver! 🎉👏"

## Series

Normally we always work on a dataframe. For the sake of completeness, however, it should be mentioned that there is a `series` data type for the individual columns. With the method `get_column` or the subset operator `[]` a column can be retrieved from a dataframe.

In [60]:
df.head(2).get_column("title")

title
str
"""Starboy"""
"""Closer"""


In [61]:
df.head(2)["title"]

title
str
"""Starboy"""
"""Closer"""


In [62]:
type(df.get_column("title"))

polars.series.series.Series

## Data types

Polars can store many different types of data in one column.

### Numbers and Boolean values

- `Int8`, `Int16`, `Int32`, `Int64`: integer number
- `Float32`, `Float64`: floating point number
- `UInt8`, `UInt16`, `UInt32`, `UInt64`: unsigned integer number
- `Decimal`: 128-bit floating point number with high precision, experimental
- `Boolean`: logical/boolean value

Numbers are created in Polars as 64-bit data types unless otherwise stated.

A column can be converted to another data type with the function 'cast', e.g. to save memory space.

In [63]:
# Default data type is Int64 or Float64 for numbers
df.select(pl.col("streams")).head(2)

streams
i64
3135625
3015525


In [64]:
# throws an error because some values are too large for Int16
try:
    df.select(pl.col("streams").cast(pl.Int16)).head(2)
except pl.exceptions.InvalidOperationError as e:
    print(e.args)


('conversion from `i64` to `i16` failed in column \'streams\' for 238 out of 238 values: [624980, 624866, … 3855766]\n\nDid not show all failed cases as there were too many.\n\nThis error occurred in the following expression:\n\tcol("streams").strict_cast(Int16)\n',)


In [65]:
# does not throw an error because Int32 is sufficiently large
df.select(pl.col("streams").cast(pl.Int32)).head(2)

streams
i32
3135625
3015525


Attention before you convert everything to the smallest possible datatype: with 32-bit datatypes overflows can occur during calculations for which no warning is issued!

In [66]:
print(f'Number of total streams with Int64 is {df.select(pl.col("streams").sum()).item()}')
print(f'Number of total streams with Int32 is {df.select(pl.col("streams").cast(pl.Int32).sum()).item()}')

Number of total streams with Int64 is 439304303921
Number of total streams with Int32 is 1217639729


With the method `shrink_dtype` the memory consumption can be reduced to some extent also in an automated fashion. However, this never changes from "signed" to "unsigned" data types, even if no negative data is present.

In [67]:
df.get_column("rank").shrink_dtype().head(2)

rank
i16
1
2


### Date and time

- `Date`: Date
- `Time`: Time
- `Datetime`: Time
- `Duration`: Time duration

You can extract components from data and times using the namespace `dt`.

In [68]:
(df
    .select("date")
    .with_columns(
        pl.col("date").dt.year().alias("year"),
        pl.col("date").dt.quarter().alias("quarter"),
        pl.col("date").dt.month().alias("month"),
        pl.col("date").dt.week().alias("week"),
        pl.col("date").dt.weekday().alias("weekday"), # Monday == 1, Sunday == 7
        pl.col("date").dt.day().alias("day"),
    )
    .sample(5)
)

date,year,quarter,month,week,weekday,day
date,i32,i8,i8,i8,i8,i8
2018-03-13,2018,1,3,11,2,13
2018-11-20,2018,4,11,47,2,20
2021-07-19,2021,3,7,29,1,19
2020-04-25,2020,2,4,17,6,25
2018-02-24,2018,1,2,8,6,24


We can also subtract dates from each other or add or subtract a period of time (`offset_by`).

In [69]:
(df
    .select("date")
    .with_columns(
        (pl.col("date").dt.month_end() - pl.col("date")).alias("days_till_month_end"),
        pl.col("date").dt.offset_by("1w").alias("same_day_next_week")
    )
    .sample(5)
)

date,days_till_month_end,same_day_next_week
date,duration[μs],date
2019-07-15,16d,2019-07-22
2018-07-04,27d,2018-07-11
2021-07-02,29d,2021-07-09
2018-04-11,19d,2018-04-18
2018-04-17,13d,2018-04-24


In [71]:
(df
    .filter(pl.col("date").eq(pl.col("date").dt.month_end()))
    .select("date", pl.col("artist"))
    .sample(5)
)

date,artist
date,str
2017-06-30,"""Justin Timberlake"""
2021-08-31,"""Elton John, Dua Lipa"""
2017-08-31,"""Rita Ora"""
2019-09-30,"""Ed Sheeran"""
2021-09-30,"""OneRepublic"""


### Character strings

- `String`: any character string
- `Categorical`: character string encoded as category

### Structures

- `List`: List with variable length per row
- `Array`: list with fixed length in all rows, e.g. coordinates
- `Struct`: named fields

### Other

- `Binary`: binary data
- `Object`: any Python object

## Sort

The `sort` method makes it easy to sort dataframes.

In [72]:
df.sort("rank").head(3)

title,rank,date,artist,url,region,chart,trend,streams
str,i64,date,str,str,str,str,str,i64
"""Starboy""",1,2017-01-01,"""The Weeknd, Daft Punk""","""https://open.spotify.com/track/5aAx2yezTd8zXrkmtKl66Z""","""Global""","""top200""","""SAME_POSITION""",3135625
"""Starboy""",1,2017-01-02,"""The Weeknd, Daft Punk""","""https://open.spotify.com/track/5aAx2yezTd8zXrkmtKl66Z""","""Global""","""top200""","""SAME_POSITION""",3342769
"""God's Plan""",1,2018-03-01,"""Drake""","""https://open.spotify.com/track/2XW4DbS6NddZxRPm5rMCeY""","""Global""","""top200""","""SAME_POSITION""",7572795


In [73]:
df.sort("streams", descending=True).head(3)

title,rank,date,artist,url,region,chart,trend,streams
str,i64,date,str,str,str,str,str,i64
"""Easy On Me""",1,2021-10-15,"""Adele""","""https://open.spotify.com/track/0gplL1WMoJ6iYaPgMCL0gX""","""Global""","""top200""","""MOVE_UP""",19749704
"""All I Want for Christmas Is You""",1,2020-12-24,"""Mariah Carey""","""https://open.spotify.com/track/0bYg9bo50gSsH3LtXe2SQn""","""Global""","""top200""","""SAME_POSITION""",17223237
"""Last Christmas""",2,2020-12-24,"""Wham!""","""https://open.spotify.com/track/2FRnf9qhLbvw8fu4IBXx78""","""Global""","""top200""","""SAME_POSITION""",15813799


In [74]:
df.sort(["rank", "streams"], descending=[False, True]).head(3)

title,rank,date,artist,url,region,chart,trend,streams
str,i64,date,str,str,str,str,str,i64
"""Easy On Me""",1,2021-10-15,"""Adele""","""https://open.spotify.com/track/0gplL1WMoJ6iYaPgMCL0gX""","""Global""","""top200""","""MOVE_UP""",19749704
"""All I Want for Christmas Is You""",1,2020-12-24,"""Mariah Carey""","""https://open.spotify.com/track/0bYg9bo50gSsH3LtXe2SQn""","""Global""","""top200""","""SAME_POSITION""",17223237
"""drivers license""",1,2021-01-15,"""Olivia Rodrigo""","""https://open.spotify.com/track/7lPN2DXiMsVn7XUKtOW1CS""","""Global""","""top200""","""SAME_POSITION""",13714177


## Write data

A dataframe can be written to a file in various formats using the `write_*` methods.

In [75]:
df_2020 = df.filter(pl.col("date").dt.year().eq(2020))

In [76]:
# as CSV (approx. 9 MB)
df_2020.write_csv("2020_write_test.csv", )

In [77]:
# as compressed CSV (approx. 2 MB)
import gzip

with gzip.open("2020_write_test.csv.gz", "wb") as f:
    df_2020.write_csv(f)

In [78]:
# as Apache Parquet (approx. 1 MB)
df_2020.write_parquet("2020_write_test.parquet")

In [79]:
!ls -l 2020_write_test*

-rw-r--r-- 1 root root 9713578 Nov 17 06:41 2020_write_test.csv
-rw-r--r-- 1 root root 2060305 Nov 17 06:41 2020_write_test.csv.gz
-rw-r--r-- 1 root root  556343 Nov 17 06:41 2020_write_test.parquet


## Optional exercises

### Question 7

In [80]:
q7.question()

Filter on all second place songs on Christmas (December 24 and 25)!


In [95]:
q7_df = df.filter(pl.col("date").dt.month().eq(12) & pl.col("date").dt.day().is_between(24,25) & pl.col("rank").eq(2))

In [96]:
q7.check(q7_df)

"✅ It might have taken a few tries, but you're unstoppable! 😅👊"

### Question 8

In [97]:
q8.question()

Calculate the average number of streams per song on Mondays and on Fridays!


In [98]:
q8_monday = df.filter(pl.col("date").dt.weekday().eq(1)).mean()
q8_friday = df.filter(pl.col("date").dt.weekday().eq(5)).mean()

In [99]:
q8.check(q8_monday, q8_friday)

"✅ Wow, first try and you nailed it! You're a natural problem-solver! 🎉👏"

### Question 9

In [100]:
q9.question()

What is the average number of days in the data set since the beginning of each month?


In [104]:
q9_df = df.select((pl.col("date") - pl.col("date").dt.month_start()).mean())

In [105]:
q9.check(q9_df)


"✅ Wow, first try and you nailed it! You're a natural problem-solver! 🎉👏"

### Question 10

In [106]:
q10.question()

Create a dataframe with all collaborations "Zedd" has been involved in.


In [108]:
q10_df = df.filter(pl.col("artist").str.contains("Zedd") & pl.col("artist").ne("Zedd")).select(pl.col("artist").unique())

In [109]:
q10.check(q10_df)

"✅ Wow, first try and you nailed it! You're a natural problem-solver! 🎉👏"

### Question 11

In [110]:
q11.question()

What is the highest chart position achieved by "Maren Morris" together with "Zedd"? And without him?


In [114]:
q11_ohne_zedd = (df.filter(pl.col("artist").str.contains("Maren Morris") & ~ pl.col("artist").str.contains("Zedd")).select(pl.col("rank").min()))

q11_mit_zedd = (df.filter(pl.col("artist").str.contains("Maren Morris") & pl.col("artist").str.contains("Zedd")).select(pl.col("rank").min()))

In [116]:
q11.check(q11_ohne_zedd, q11_mit_zedd)


q11_without_zedd = (
    df.filter(
        pl.col("artist").str.contains("Maren Morris") &
        ~ pl.col("artist").str.contains("Zedd")
    )
    .select(pl.col("rank").min())
)

q11_with_zedd = (
    df.filter(
        pl.col("artist").str.contains("Maren Morris") &
        pl.col("artist").str.contains("Zedd")
    )
    .select(pl.col("rank").min())
)
    


### Question 12

In [117]:
q12.question()


Minimize the memory consumption of the dataframe by using different data types and removing an unnecessary prefix.
You can display the memory consumption with df.estimated_size("mb").
    


In [131]:
q12_df = df.with_columns(pl.col("title").cast(pl.Categorical),pl.col("artist").cast(pl.Categorical),pl.col("trend").cast(pl.Categorical),pl.col("region").cast(pl.Categorical),pl.col("rank").cast(pl.UInt8),pl.col("chart").cast(pl.Categorical),pl.col("streams").cast(pl.UInt32),pl.col("url").cast(pl.Categorical))

In [132]:
q12.check(df, q12_df)

'✅ Right on target! You hit the bullseye on your second shot! 🎯👏'